In [1]:
!pip install sympy


[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import sympy as sp

t = 1e-8
x, y = sp.symbols("x y", real=True)

lam = sp.symbols("lambda", real=True)
f = 2 * x - y
constraints = [
    x**2 + y**2 - 5
]

variables = [x, y]
multipliers = [lam]

J = sp.Matrix(constraints).jacobian(variables)

print("Матрица Якоби ограничений:")
sp.pprint(J)

singular_solutions = sp.solve(
    constraints + list(J),
    variables,
    dict=True
)

if not singular_solutions:
    print("Особых допустимых точек нет.")
    print("Все допустимые точки регулярные.")
else:
    print("Найдены особые допустимые точки:")
    print(singular_solutions)

L = f + sum(
    a * h
    for a, h in zip(multipliers, constraints)
)

equations = [
    sp.diff(L, v)
    for v in variables
] + constraints

unknowns = variables + multipliers

solutions = sp.solve(
    equations,
    unknowns,
    dict=True
)

print("Целевая функция:")
sp.pprint(f)

print("\nОграничения:")
for h in constraints:
    sp.pprint(h)

print("\nФункция Лагранжа:")
sp.pprint(L)

print("\nУравнения стационарности:")
for eq in equations:
    sp.pprint(eq)

print("\nНайденные кандидаты:")

valid_solutions = []

for number, solution in enumerate(solutions, start=1):

    if not all(
        sp.ask(sp.Q.real(solution[v])) is True
        for v in unknowns
    ):
        continue

    point_subs = {
        v: solution[v]
        for v in variables
    }

    residuals = [
        sp.simplify(h.subs(point_subs))
        for h in constraints
    ]

    is_feasible = all(
        abs(float(r)) <= t
        for r in residuals
    )

    value = sp.simplify(f.subs(point_subs))

    print(f"\n{'─' * 35}")
    print(f"Кандидат {number}")
    print(f"{'─' * 35}")

    print("Точка:", tuple(solution[v] for v in variables))
    print("Множитель Лагранжа:", solution[lam])
    print("Невязки:", residuals)
    print("Допустимость:", "да" if is_feasible else "нет")
    print("Значение функции:", value)
    if is_feasible:
        valid_solutions.append(solution)


Матрица Якоби ограничений:
[2⋅x  2⋅y]
Особых допустимых точек нет.
Все допустимые точки регулярные.
Целевая функция:
2⋅x - y

Ограничения:
 2    2    
x  + y  - 5

Функция Лагранжа:
  ⎛ 2    2    ⎞          
λ⋅⎝x  + y  - 5⎠ + 2⋅x - y

Уравнения стационарности:
2⋅λ⋅x + 2
2⋅λ⋅y - 1
 2    2    
x  + y  - 5

Найденные кандидаты:

───────────────────────────────────
Кандидат 1
───────────────────────────────────
Точка: (2, -1)
Множитель Лагранжа: -1/2
Невязки: [0]
Допустимость: да
Значение функции: 5

───────────────────────────────────
Кандидат 2
───────────────────────────────────
Точка: (-2, 1)
Множитель Лагранжа: 1/2
Невязки: [0]
Допустимость: да
Значение функции: -5


In [2]:
values = []

for solution in valid_solutions:
    point = {
        x: solution[x],
        y: solution[y]
    }

    value = sp.simplify(f.subs(point))

    values.append({
        "point": (solution[x], solution[y]),
        "value": value
    })

min_candidate = min(values, key=lambda item: item["value"])
max_candidate = max(values, key=lambda item: item["value"])

print("\nИтог:")
print("Глобальный минимум:",
      min_candidate["value"],
      "в точке",
      min_candidate["point"])

print("Глобальный максимум:",
      max_candidate["value"],
      "в точке",
      max_candidate["point"])


Итог:
Глобальный минимум: -5 в точке (-2, 1)
Глобальный максимум: 5 в точке (2, -1)


### Обоснование глобальности экстремумов

Рассматривается задача оптимизации функции

$$
f(x,y)=2x-y
$$

при ограничении

$$
x^2+y^2=5.
$$

**1. Компактность допустимого множества**

Допустимое множество

$$
D=\{(x,y)\in\mathbb{R}^2 \mid x^2+y^2=5\}
$$

представляет собой окружность радиуса $\sqrt{5}$.

Окружность является замкнутым и ограниченным множеством,
следовательно, она компактна.

**2. Непрерывность целевой функции**

Функция $f(x,y)=2x-y$ является линейной и непрерывной
на всей плоскости $\mathbb{R}^2$.

По теореме Вейерштрасса непрерывная функция на компактном
множестве достигает глобального минимума и максимума.

**3. Полнота найденных кандидатов**

Все допустимые точки регулярны, поскольку

$$
\nabla h(x,y)=(2x,2y)\neq(0,0)
$$

на окружности $x^2+y^2=5$.

При решении системы Лагранжа найдены все действительные
точки-кандидаты:

$$
P_1=(-2,1), \qquad P_2=(2,-1).
$$

**4. Сравнение значений функции**

$$
f(-2,1)=2(-2)-1=-5,
$$

$$
f(2,-1)=2(2)-(-1)=5.
$$

Следовательно, глобальные экстремумы:

- **Глобальный минимум:** $f_{\min}=-5$ в точке $(-2,1)$.
- **Глобальный максимум:** $f_{\max}=5$ в точке $(2,-1)$.

Таким образом, глобальность найденных экстремумов
обоснована компактностью допустимого множества,
непрерывностью целевой функции и полнотой списка кандидатов.